In [83]:
import plotly.graph_objects as go
import json
import pandas as pd

In [84]:
base_path = "../prompt_structure_results"
score_path = "../prompt_eval_results"
split = "test"
template = "Instruct-Query"
use_lang_specific_prompts=False
k = 10
models = [#"minishlab__potion-base-8M",
          #"google__embeddinggemma-300m",          
          #"intfloat__multilingual-e5-small",
          #"BAAI__bge-m3",
          "intfloat__multilingual-e5-large-instruct",
          "Qwen__Qwen3-Embedding-0.6B",
          "microsoft__harrier-oss-v1-0.6b",
          ]

dataset = "mteb__reddit-clustering"
path = lambda model: f"{base_path}/{model}/{dataset}/{split}/{template}_template/"
path_scores = lambda model: f"{score_path}/{model}/{dataset}/{split}/{template}_template/"

In [85]:
"""
def prompts(data_name):
    if not use_lang_specific_prompts:
        return {"arcchallenge": get_prompts_arcchallenge(),
                "tatoeba": get_prompts_tatoeba(),
                "summeval-2": get_prompts_summeval(),
                "webfaq": get_prompts_webfaq()}[data_name]
    else:
        if data_name == "tatoeba":
            return get_prompts_tatoeba(lang=lang)
        elif data_name == "webfaq":
            return get_prompts_webfaq(lang=lang)
        else:
            return {"arcchallenge": get_prompts_arcchallenge(),
                "summeval-2": get_prompts_summeval()}[data_name]
"""


prompts_arcchallenge = {
    "NO_PROMPT":2,
    # Good, on-task prompts (varying length)
    "Given a science question, retrieve the passage that best answers it":1,
    "Retrieve the passage that correctly answers the multiple choice science question":1,
    "Find the document that contains the answer to the given science question":1,
    "Given a challenging science question, retrieve a passage that provides the correct answer and explains the underlying concept":1,
    "Retrieve passages that answer elementary and middle school science questions spanning biology, chemistry, physics, and earth science":1,
    "Find the most relevant scientific passage that directly answers the question":1,
    "Given a multiple choice question about science, retrieve the passage most likely to contain the correct answer":1,
    "Retrieve a passage answering the question":1,
    "Find relevant science passages":1,
    "Answer the science question":1,
    "Retrieve the correct answer":1,

    # Slightly off or vague prompts
    "Find documents related to the topic":0.5,
    "Retrieve a relevant passage":0.5,
    "Given a question, find a related document":0.5,
    "Search for information about the query":0.5,
    "Look up the answer":0.5,
    "Find something useful":0.5,
    "Get the relevant text":0.5,
    "Retrieve":0.5,

    # Wrong task prompts
    "Translate the following sentence into French":0,
    "Summarize the given paragraph into two sentences":0,
    "Given two sentences, determine if they are semantically similar":0,
    "Classify the sentiment of the given review as positive or negative":0,
    "Generate a creative story based on the prompt":0,
    "Retrieve duplicate questions from the forum":0,
    "Find the most similar product review":0,
    "Given a code snippet, retrieve the documentation":0,

    # Nonsense / keysmash prompts
    "asdfjkl qpwoeiru zxcvbnm":-1,
    "hgJKSbf oiawnef LKJHDS kdjfbs":-1,
    "!!!! ??? ### @@@":-1,
}

prompts_tatoeba = {
    "NO_PROMPT":2,
    # Well-Fitting Instructions
    "Retrieve the corresponding translation":1,
    "Find the equivalent sentence in another language":1,
    "Retrieve a parallel sentence":1,
    "Given an English sentence, find its translation":1,
    "Match the English sentence to its translated counterpart in the target language":1,
    "Identify the sentence in the target language that is a direct translation of the given English source sentence":1,
    "Retrieve":1,
    "Find translation":1,
    "Given the following English text, retrieve the passage that is a faithful and accurate translation of it into a non-English language":1,
    "Translate":1,

    # Related but Suboptimal Instructions
    "Find a sentence that has similar meaning":0.5,
    "Retrieve a semantically similar passage":0.5,
    "Find a paraphrase of the given sentence":0.5,
    "Retrieve the most thematically related passage from the document corpus":0.5,
    "Find a sentence that discusses the same topic":0.5,
    "Given a query sentence, retrieve passages that are topically relevant":0.5,
    "Identify the passage that best captures the intent of the following sentence":0.5,
    "Retrieve the passage that contains overlapping vocabulary with the source sentence":0.5,

    # Wrong Task Instructions
    "Classify the sentiment of the given review as positive or negative":0,
    "Summarize the following paragraph in two sentences":0,
    "Answer the following question based on the provided context":0,
    "Extract all named entities from the following sentence and label them as PERSON, ORGANIZATION, or LOCATION":0,
    "Determine whether the following two statements are contradictory, entailed, or neutral":0,
    "Generate a creative short story based on the following prompt":0,
    "Given the following code snippet, identify and fix the bug":0,

    # Nonsense / keysmash prompts
    "asdfjkl qpwoeiru zxcvbnm":-1,
    "hgJKSbf oiawnef LKJHDS kdjfbs":-1,
    "!!!! ??? ### @@@":-1,
}

prompts_summeval = {
    "NO_PROMPT":2,
    # Well-Fitting Instructions
    "Given a summary, retrieve the full document it was derived from":1,
    "Match the short abstract to the longer document it summarises":1,
    "Retrieve the document whose key points are captured in the given summary":1,
    "Given the following brief summary, find the source document that contains the information it describes":1,
    "Retrieve":1,
    "Given a condensed version of a text, retrieve the original document from which it was summarised":1,
    "Identify the passage that the following summary is an abridged version of":1,

    # Related but Suboptimal Instructions
    "Retrieve a document that discusses the same subject matter":0.5,
    "Find a passage that is thematically related to the given text":0.5,
    "Retrieve the most topically similar document":0.5,
    "Find a passage that covers overlapping key concepts":0.5,
    "Given a short text, retrieve a longer passage on the same topic":0.5,
    "Identify the document that shares the most content words with the given passage":0.5,
    "Retrieve a passage that is semantically close to the given description":0.5,
    "Find a document that would be relevant to someone interested in this topic":0.5,

    # Wrong Task Instructions, first 3 are the wrong way round
    "Find summary":0,
    "Retrieve the document that best summarises the given topic":0,
    "Find the passage that serves as a concise summary of the source text":0,
    "Translate the following sentence into French":0,
    "Answer the following question using the provided context":0,
    "Classify the sentiment of the following customer review":0,
    "Extract all named entities from the text and categorise them as PERSON, LOCATION, or ORGANIZATION":0,
    "Determine whether the following two passages are contradictory, entailed, or neutral":0,
    "Generate a creative short story inspired by the following prompt":0,
    "Given the following code, identify the bug and suggest a fix":0,

    # Nonsense / keysmash prompts
    "asdfjkl qpwoeiru zxcvbnm":-1,
    "hgJKSbf oiawnef LKJHDS kdjfbs":-1,
    "!!!! ??? ### @@@":-1,
}

prompts_webfaq = {
        "NO_PROMPT":2,
        # Good, on-task prompts (varying length and specificity)
        "Given a question, retrieve the passage that correctly answers it":1,
        "Retrieve the passage that answers the following question":1,
        "Find the document that contains the answer to this question":1,
        "Given a question, retrieve the most relevant passage that directly provides the answer":1,
        "Retrieve a passage that answers the given question accurately and completely":1,
        "Find the best-matching answer passage for the following question":1,
        "Given a question, find the paragraph from which the answer can be extracted":1,
        "Given a question, retrieve the passage that answers it":1,
        "Retrieve the passage containing the answer to the question":1,
        "Find the answer passage for the given question":1,
        "Retrieve the most relevant passage that answers the query":1,
        "Answer the question":1,

        # Slightly off or vague prompts
        "Find documents related to the topic of the question":0.5,
        "Retrieve a relevant passage":0.5,
        "Given a query, find a related document":0.5,
        "Search for information":0.5,
        "Look up the answer to the question":0.5,
        "Find something relevant":0.5,
        "Get a passage that might be useful":0.5,
        "Retrieve":0.5,

        # Wrong task prompts
        "Translate the following sentence from French into English":0,
        "Summarize the given paragraph into a few sentences":0,
        "Given two sentences, determine if they are semantically similar":0,
        "Classify the sentiment of the given review as positive or negative":0,
        "Generate a fluent continuation of the following paragraph":0,
        "Given a premise and a hypothesis, determine the entailment relationship":0,
        "Retrieve duplicate questions from the forum":0,

        # Nonsense / keysmash prompts
        "asdfjkl qpwoeiru zxcvbnm":-1,
        "hgJKSbf oiawnef LKJHDS kdjfbs":-1,
        "!!!! ??? ### @@@":-1
}


def prompts(dataset):
    return {"arcchallenge": prompts_arcchallenge,
            "tatoeba:fin-eng" : prompts_tatoeba,
            "summeval-2": prompts_summeval,
            "webfaq:deu": prompts_webfaq,
            "webfaq:eng": prompts_webfaq}[dataset]

In [86]:

def construct_df(model, show=False):
    scores_path= path_scores(model)+f"results@{k}.json"
    with open(scores_path) as f:
        scores = json.load(f)
    with open(path(model)+"prompt_geometry.json") as f:
        data1 = json.load(f)
    if show:
        print(data1.keys())
        #print(data1)
    print(scores)
    df_scores = pd.DataFrame.from_dict(scores).T#, orient="index", columns=["score"])
    #columns= [f"prompt{i}" for i in range(len(scores.values()))])
    #df_scores = df_scores.reset_index().rename(columns={"index": "prompt_text"})#, "mean":"score_mean", "std":"score_std"})
    if show: display(df_scores.head())
    #df_scores["score_mean"] = pd.to_numeric(df_scores["score_mean"])
    df_angle = pd.DataFrame.from_dict(data1).T
    #display(df_scores.head())
    if show: display(df_angle.head())
    df = df_scores.merge(df_angle, on='prompt_text')
    #prompt_dict = prompts(dataset)
    #df["prompt_label"] = df["prompt_text"].apply(lambda prompt: prompt_dict[prompt])
    if show: display(df.head())
    return df

_ = construct_df(models[0], show=False)

{'prompt0': {'prompt_text': 'NO_PROMPT', 'V-score': 0.6011537019910127, 'AMI': 0.5997549206158485, 'Accuracy': 0.8850430914651098, 'F1': 0.8850430914651098}, 'prompt1': {'prompt_text': 'EMPTY', 'V-score': 0.649379220276302, 'AMI': 0.6481428596680183, 'Accuracy': 0.880038921323325, 'F1': 0.880038921323325}, 'prompt2': {'prompt_text': 'Identify the main topic of this text for grouping similar documents together.', 'V-score': 0.5142053874134601, 'AMI': 0.5124947785908773, 'Accuracy': 0.8509869335557408, 'F1': 0.8509869335557408}, 'prompt3': {'prompt_text': 'Represent this sentence for clustering with other semantically similar sentences.', 'V-score': 0.458515272575338, 'AMI': 0.456618258530076, 'Accuracy': 0.8729496802891298, 'F1': 0.8729496802891298}, 'prompt4': {'prompt_text': 'Summarize the core theme of the following passage so it can be grouped with related texts.', 'V-score': 0.5475746676755818, 'AMI': 0.5459874631924123, 'Accuracy': 0.869752571587434, 'F1': 0.869752571587434}, 'pro

In [87]:

def plot(df, x, y="score", colors=None, sizes=None, title="", legend_title=None):
    if colors is None:
        colors = y
    if sizes is None:
        sizes = y
    
    # legend title that explains formatting
    if legend_title is None:
        legend_title = f"colors:{colors}, size:{sizes}"

    # Normalize scores for marker size
    min_size, max_size = 10, 30
    try:
        # parse the value from dictionary
        df["sizes"] = df[sizes].apply(lambda d: float(d['mean']))
        ranks = df["sizes"].rank(method='average')
    except:   # for non-dict format: i.e. prompt_label or score
        ranks = df[sizes].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )

    y_vals = df[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_err = df[y].apply(lambda d: float(d['std']) if isinstance(d, dict) else float(d[1]) if isinstance(d, list) else 0.0)

    x_vals = df[x].apply(lambda d: float(d['mean']))
    x_err  = df[x].apply(lambda d: float(d['std']))


    # Create figure
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            mode='markers',
            x=x_vals,
            y=y_vals,
            error_x=dict(type='data', array=x_err, visible=True, color='lightgray'),  # optional std bars
            error_y=dict(type='data', array=y_err, visible=True, color='lightgray'),
            marker=dict(
                size=marker_sizes,
            #    colorscale='Cividis',
                color=df[colors].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d)),
                colorbar=dict(title=f"C:{colors} S:{sizes}"),
                showscale=True,
            ),
            text=df['prompt_text'],
            hovertemplate=(
                '<b>Prompt:</b> %{text}<br>'
                '<b>X (distance):</b> %{x:.4f}<br>'
                '<b>Y (score):</b> %{y:.4f}<br>'
            ),
        ),
    )


    # Add one trace per alpha value

    fig.update_layout(
        title = title,
        xaxis_title=x,#'Cos-distance compared to Q-A line',
        yaxis_title=y,#'Prompt performance',
        height=600,
        width=1000,
        template="none",
    )
    fig.update_layout(legend_title_text=legend_title)


    fig.show()

In [88]:
dfs = {}
for m in models:
    dfs[m] = construct_df(m)

{'prompt0': {'prompt_text': 'NO_PROMPT', 'V-score': 0.6011537019910127, 'AMI': 0.5997549206158485, 'Accuracy': 0.8850430914651098, 'F1': 0.8850430914651098}, 'prompt1': {'prompt_text': 'EMPTY', 'V-score': 0.649379220276302, 'AMI': 0.6481428596680183, 'Accuracy': 0.880038921323325, 'F1': 0.880038921323325}, 'prompt2': {'prompt_text': 'Identify the main topic of this text for grouping similar documents together.', 'V-score': 0.5142053874134601, 'AMI': 0.5124947785908773, 'Accuracy': 0.8509869335557408, 'F1': 0.8509869335557408}, 'prompt3': {'prompt_text': 'Represent this sentence for clustering with other semantically similar sentences.', 'V-score': 0.458515272575338, 'AMI': 0.456618258530076, 'Accuracy': 0.8729496802891298, 'F1': 0.8729496802891298}, 'prompt4': {'prompt_text': 'Summarize the core theme of the following passage so it can be grouped with related texts.', 'V-score': 0.5475746676755818, 'AMI': 0.5459874631924123, 'Accuracy': 0.869752571587434, 'F1': 0.869752571587434}, 'pro

In [94]:

for m in models:
    df = dfs[m]
    score="V-score" #"ndcg@10"
    #print(df.columns)
    #plot(df, "displacement", y=score, title=f"{dataset}: {m}: Does more movement(x) mean better score(y)")
    plot(df, "sim_improvement", y=score, title=f"{dataset}: {m}: Does sim-improvement (x) actually mean better score? (y)")
    #plot(df, "chord_similarity", y=score, title=f"{dataset}: {m}: Fraction of change toward Answer (x) vs. performance(y).")
    #plot(df, "knn_retention", y=score,title=f"{dataset}: {m}: Neighborhood retention (x) vs. performance (y)")
    #plot(df, "orthogonal_magnitude", y=score, title=f"{dataset}: {m}:")
    #plot(df, "hard_neg_angulation", y=score,title=f"{dataset}: {m}: Angle from false negs vs. evaluation score")
    #plot(df, "hard_neg_sim_change", y=score,title=f"{dataset}: {m}: Movement away from false negs vs. evaluation score")#, sizes="sim_improvement", colors="sim_improvement")
    
	# combinations
    #plot(df, "hard_neg_sim_change", y="sim_improvement", sizes="score", colors="knn_retention",title=f"{dataset}: {m}:")
    #plot(df, "hard_neg_angulation", y="sim_improvement", sizes="score", colors="score",title=f"{dataset}: {m}: Angle from false negs vs. sim improvement")
    #plot(df, "displacement", y="sim_improvement", colors="knn_retention", sizes="knn_retention", title=f"{dataset}: {m}: Movement (x) vs. similarity improvement(y).")
    #plot(df, "parallel_fraction", y="sim_improvement", title=f"{dataset}: {m}: Fraction of change toward Answer (x) vs. similarity gains.")
    #plot(df, "displacement", y="parallel_fraction", sizes="score", colors="prompt_label", title=f"{dataset}: {m}: Does more movement(x) mean better score(y)")





| Metric	| Question it answers | Value meanings |
|--------|--------|--------|
|chord_similarity	|Does the prompt push q in the same direction as a?| large = yes, small = no |
|sim_improvement	|Does the prompt make pq closer to a? (the practical question)| pos = prompt did move us closer|
|displacement	|How much does the prompt change the embedding?| large = we moved a lot |
|parallel_magnitude	|How much movement is toward the answer? | negative = away, positive = towards |
|orthogonal_magnitude	|How much movement is sideways (perpendicular to the q→a axis)?| large = a lot, small = a little |
|parallel_fraction	|What fraction of total movement goes toward the answer?| large = a lot towards the answer |
|knn_retention | Does the prompt unify structure? | large = yes, small = no |
|hard_neg_sim | For the closest incorrect, how much did we move away? | small = we moved, large = we did not|

